In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Runs in the yolo-train conda environment; additionally requires:
# pip install shapely
'''

In [ ]:
# === Cell 3: Imports ===

%matplotlib inline

import sys
import time
import random
import platform
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import shapely
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import distance_transform_edt, label as nd_label

import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams["figure.dpi"]    = 120
plt.rcParams["savefig.dpi"]   = 150
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# === Cell 4: Config ===

PROJECT_NAME = "20260622_Measurement_Point_Sampling"

# --- Paths ---
PROJECT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/03_Measurement_Point_Sampling")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Label sources; both contain YOLO segmentation .txt files.
PREDICTED_LABELS_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260603_Yolo_Deployment_Training/labels")
GT_LABELS_DIR        = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/labels/all")

# Source images used as widget background; set to None if unavailable.
IMAGE_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Initial_Image_Dataset/1080_resolution")

# --- Image dimensions ---
IMG_W = 1920
IMG_H = 1080

# --- Sensor calibration ---
# px/cm ratio measured on a 1080p reference image; the sensor footprint
# radius [px] follows from the physical spot size.
CALIB_PIXELS       = 355          # pixels measured for the reference length
CALIB_CM           = 10.0         # physical reference length [cm]
PIXELS_PER_CM      = CALIB_PIXELS / CALIB_CM        # -> 35.5 px/cm

SENSOR_DIAMETER_CM = 2.5          # physical sensor spot diameter [cm]
SENSOR_DIAMETER_PX = SENSOR_DIAMETER_CM * PIXELS_PER_CM
SENSOR_RADIUS_PX   = round(SENSOR_DIAMETER_PX / 2)  # -> round(44.375) = 44 px

# --- Minimum fragment area ---
# Fragments below this valid area [px²] are ignored.
MIN_VALID_AREA_PX2 = 1000

# --- Sampling algorithms ---
#   "CENTROID": area-weighted centroid with nearest-valid-pixel fallback
#   "EDT_PEAK": Euclidean distance-transform peak
ALGORITHMS = ["CENTROID", "EDT_PEAK"]

# --- Line scanner grid ---
# LINE_SPACING_Y_PX: distance between scan lines along the conveyor [px]
# LINE_SPACING_X_PX: spacing between spectra along one scan line [px]
LINE_SPACING_Y_PX = 40
LINE_SPACING_X_PX = 40

# --- Class mapping ---
CLASS_MAP = {
    "body":  0,
    "cap":   1,
    "label": 2,
}
CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

# --- Reproducibility ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [ ]:
# === Cell 5: Environment and Path Checks ===

print("Environment")
print("-" * 72)
print(f"Python:    {sys.version}")
print(f"Platform:  {platform.platform()}")
print(f"NumPy:     {np.__version__}")
print(f"OpenCV:    {cv2.__version__}")
print(f"Shapely:   {shapely.__version__}")

print()
print("Path checks")
print("-" * 72)

def _check(label, path, kind="dir"):
    ok  = path.is_dir() if kind == "dir" else path.is_file()
    tag = "DIR " if kind == "dir" else "FILE"
    print(f"  [{tag}] {'OK' if ok else 'MISSING':8s} {label}: {path}")
    return ok

all_ok = True
all_ok &= _check("PROJECT_ROOT",         PROJECT_ROOT,         "dir")
all_ok &= _check("RESULTS_DIR",          RESULTS_DIR,          "dir")
all_ok &= _check("PREDICTED_LABELS_DIR", PREDICTED_LABELS_DIR, "dir")
all_ok &= _check("GT_LABELS_DIR",        GT_LABELS_DIR,        "dir")

if IMAGE_DIR is not None:
    ok_img = _check("IMAGE_DIR", IMAGE_DIR, "dir")
    if not ok_img:
        print("  IMAGE_DIR missing — blank canvas will be used in the widget.")
        IMAGE_DIR = None

# Label file counts per source
for lbl, d in [("Predicted", PREDICTED_LABELS_DIR), ("GT", GT_LABELS_DIR)]:
    if d.exists():
        n = len(list(d.glob("*.txt")))
        print(f"  {lbl} labels: {n} .txt file(s)")

print()
print(f"Config summary")
print("-" * 72)
print(f"  IMG_W x IMG_H:       {IMG_W} x {IMG_H}")
print(f"  SENSOR_RADIUS_PX:    {SENSOR_RADIUS_PX}")
print(f"  MIN_VALID_AREA_PX2:  {MIN_VALID_AREA_PX2}")
print(f"  ALGORITHMS:          {ALGORITHMS}")
print()
print("All paths OK." if all_ok else "WARNING: Some paths missing — see above.")

In [ ]:
# === Cell 6: Data Loader ===

def parse_label_file(txt_path: Path, img_w: int = IMG_W, img_h: int = IMG_H) -> list:
    """
    Parse one YOLO segmentation .txt file.

    Line format: class_id x1 y1 x2 y2 ... xN yN (coordinates normalised to [0, 1]).

    Returns a list of dicts with class_id, polygon_norm and polygon_px.
    """
    instances = []
    if not txt_path.is_file():
        return instances
    with txt_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            cls_id = int(float(parts[0]))
            coords = [float(v) for v in parts[1:]]
            if len(coords) % 2 != 0:
                coords = coords[:-1]
            poly_norm = list(zip(coords[0::2], coords[1::2]))
            poly_px   = [
                (int(round(x * img_w)), int(round(y * img_h)))
                for x, y in poly_norm
            ]
            instances.append({
                "class_id":     cls_id,
                "polygon_norm": poly_norm,
                "polygon_px":   poly_px,
            })
    return instances


def load_all_labels(input_dir: Path) -> dict:
    """Load all .txt files from input_dir. Returns {stem: [instance_dicts]}."""
    data = {}
    for txt_path in sorted(input_dir.glob("*.txt")):
        data[txt_path.stem] = parse_label_file(txt_path)
    return data


# Load both label sources
LABELS_PRED = load_all_labels(PREDICTED_LABELS_DIR)
LABELS_GT   = load_all_labels(GT_LABELS_DIR)

# Stems available in either source
ALL_STEMS = sorted(set(LABELS_PRED.keys()) | set(LABELS_GT.keys()))

print(f"Predicted labels: {len(LABELS_PRED)} files")
print(f"GT labels:        {len(LABELS_GT)} files")
print(f"Union stems:      {len(ALL_STEMS)}")

In [ ]:
# === Cell 7: Mask Builder ===

def rasterise_polygon(poly_px: list, h: int = IMG_H, w: int = IMG_W) -> np.ndarray:
    """Rasterise a pixel-space polygon to a boolean (H x W) mask."""
    mask = np.zeros((h, w), dtype=bool)
    if len(poly_px) < 3:
        return mask
    pts = np.array(poly_px, dtype=np.int32).reshape(-1, 1, 2)
    canvas = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(canvas, [pts], 1)
    return canvas.astype(bool)


def build_masks(instances: list, h: int = IMG_H, w: int = IMG_W) -> dict:
    """
    Build per-instance valid masks.

    Body validity:  valid_body_i = body_mask_i AND NOT union(all label masks)
    Cap / label:    full interior is valid.

    Returns dict with keys 'body', 'cap', 'label' (each a list of entry dicts)
    and 'label_union'.
    """
    body_insts  = [i for i in instances if i["class_id"] == 0]
    cap_insts   = [i for i in instances if i["class_id"] == 1]
    label_insts = [i for i in instances if i["class_id"] == 2]

    label_masks_raw = [rasterise_polygon(i["polygon_px"], h, w) for i in label_insts]
    label_union     = np.zeros((h, w), dtype=bool)
    for lm in label_masks_raw:
        label_union |= lm

    body_entries = []
    for inst in body_insts:
        raw   = rasterise_polygon(inst["polygon_px"], h, w)
        valid = raw & ~label_union
        body_entries.append({"raw": raw, "valid": valid, "polygon_px": inst["polygon_px"]})

    cap_entries = []
    for inst in cap_insts:
        raw = rasterise_polygon(inst["polygon_px"], h, w)
        cap_entries.append({"raw": raw, "valid": raw.copy(), "polygon_px": inst["polygon_px"]})

    label_entries = []
    for inst, raw in zip(label_insts, label_masks_raw):
        label_entries.append({"raw": raw, "valid": raw.copy(), "polygon_px": inst["polygon_px"]})

    return {
        "body": body_entries, "cap": cap_entries, "label": label_entries,
        "label_union": label_union,
    }


def get_connected_components(valid_mask: np.ndarray, min_area: int = MIN_VALID_AREA_PX2):
    """Connected components of valid_mask above min_area, sorted desc by area."""
    if valid_mask.sum() == 0:
        return []
    labeled, n_comp = nd_label(valid_mask)
    components = []
    for i in range(1, n_comp + 1):
        comp_mask = labeled == i
        area = int(comp_mask.sum())
        if area >= min_area:
            components.append((area, comp_mask))
    components.sort(key=lambda x: x[0], reverse=True)
    return [m for _, m in components]


print("Module 2 (mask builder) ready.")

In [ ]:
# === Cell 8: Sampler Implementations ===
# Interface: sample(valid_mask, params) -> list of result dicts, one per fragment
# >= min_area, with keys point, score, area_px2, is_main, valid, algorithm.

def _score_at_point(edt: np.ndarray, pt):
    x, y = pt
    x = int(np.clip(x, 0, edt.shape[1] - 1))
    y = int(np.clip(y, 0, edt.shape[0] - 1))
    return float(edt[y, x])


def _nearest_valid_pixel(valid_mask: np.ndarray, pt):
    ys, xs = np.where(valid_mask)
    if len(ys) == 0:
        return None
    px, py = int(round(pt[0])), int(round(pt[1]))
    dists  = (xs - px) ** 2 + (ys - py) ** 2
    idx    = int(np.argmin(dists))
    return (int(xs[idx]), int(ys[idx]))


def _no_result(alg):
    return [{
        "point": None, "score": 0.0, "area_px2": 0,
        "is_main": True, "valid": False, "algorithm": alg,
    }]


def _crop_to_bbox(mask: np.ndarray):
    """
    Return (cropped_mask, row_offset, col_offset) for the tight bounding box.

    Cropping before distance_transform_edt reduces the array from full-image
    size to the fragment footprint, as in the live demonstrator. Crop-local
    points are shifted by (col_offset, row_offset) to full-image coordinates.
    Both samplers use this helper so their timings are comparable.
    """
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any():
        return mask, 0, 0
    r0 = int(np.argmax(rows))
    r1 = int(len(rows) - 1 - np.argmax(rows[::-1]))
    c0 = int(np.argmax(cols))
    c1 = int(len(cols) - 1 - np.argmax(cols[::-1]))
    H, W = mask.shape
    crop = mask[r0:r1 + 1, c0:c1 + 1]
    # Pad a 1 px False ring on every side not at the image border, so the EDT
    # sees the true region boundary; sides at the border stay unpadded. Results
    # are identical to the full-frame EDT; the offsets include the padding.
    pt_top    = 1 if r0 > 0     else 0
    pt_bottom = 1 if r1 < H - 1 else 0
    pt_left   = 1 if c0 > 0     else 0
    pt_right  = 1 if c1 < W - 1 else 0
    padded = np.pad(
        crop, ((pt_top, pt_bottom), (pt_left, pt_right)),
        mode="constant", constant_values=False,
    )
    return padded, r0 - pt_top, c0 - pt_left


# --- Algorithm: CENTROID ---
def sample_centroid(valid_mask: np.ndarray, params: dict) -> list:
    """
    Area-weighted centroid with boundary fallback.

    Computes the mean (x, y) of all valid pixels per fragment. If the centroid
    lies outside the valid mask (concave or annular shapes after label
    subtraction), it is projected to the nearest valid pixel.
    Score = EDT value at the returned point = clearance radius [px].

    Ref: Stoppi et al., US 12,172,310 B2 (2024).
    """
    min_area  = params.get("min_area", MIN_VALID_AREA_PX2)
    fragments = get_connected_components(valid_mask, min_area)
    if not fragments:
        return _no_result("CENTROID")

    results = []
    for i, frag in enumerate(fragments):
        frag_crop, r0, c0 = _crop_to_bbox(frag)
        edt    = distance_transform_edt(frag_crop).astype(float)
        area   = int(frag_crop.sum())
        ys, xs = np.where(frag_crop)
        cx, cy = float(xs.mean()), float(ys.mean())   # crop-local float centroid

        # Round in full-image space (round-half-to-even is not invariant under
        # integer offsets), then map back to a crop-local index for the
        # membership test and EDT lookup; matches the un-cropped result.
        fx, fy = int(round(cx + c0)), int(round(cy + r0))
        ix = int(np.clip(fx - c0, 0, frag_crop.shape[1] - 1))
        iy = int(np.clip(fy - r0, 0, frag_crop.shape[0] - 1))
        if not frag_crop[iy, ix]:
            # Fallback: nearest valid pixel to the full-space rounded centroid.
            qx, qy  = float(fx - c0), float(fy - r0)
            pt_crop = _nearest_valid_pixel(frag_crop, (qx, qy))
            pt      = (pt_crop[0] + c0, pt_crop[1] + r0) if pt_crop else None
            score   = _score_at_point(edt, pt_crop) if pt_crop else 0.0
        else:
            pt_crop = (ix, iy)
            pt      = (fx, fy)
            score   = _score_at_point(edt, pt_crop)

        results.append({
            "point": pt, "score": round(score, 2), "area_px2": area,
            "is_main": (i == 0), "valid": pt is not None, "algorithm": "CENTROID",
        })
    return results


# --- Algorithm: EDT_PEAK ---
def sample_edt_peak(valid_mask: np.ndarray, params: dict) -> list:
    """
    Euclidean distance-transform peak.

    One distance_transform_edt call per fragment provides:
      - Erosion safe-zone:  eroded = {EDT >= sensor_radius}
      - Chebyshev centre:   argmax(EDT) = largest-inscribed-circle centre
      - DT peak:            EDT value = clearance radius = quality score

    Ties on symmetric shapes are resolved by the centroid of the top-1% plateau.

    Refs: Felzenszwalb & Huttenlocher, Theory of Computing 2012,
          DOI 10.4086/toc.2012.v008a019.
          Garcia-Castellanos & Lombardo, Scot. Geog. J. 2007,
          DOI 10.1080/14702540801897809.
          Schwarz et al., ICRA 2018, DOI 10.1109/ICRA.2018.8461195.
          Bacchin et al. (WasteGAN), arXiv:2409.16999.
    """
    min_area  = params.get("min_area", MIN_VALID_AREA_PX2)
    fragments = get_connected_components(valid_mask, min_area)
    if not fragments:
        return _no_result("EDT_PEAK")

    results = []
    for i, frag in enumerate(fragments):
        frag_crop, r0, c0 = _crop_to_bbox(frag)
        edt      = distance_transform_edt(frag_crop).astype(float)
        area     = int(frag_crop.sum())
        peak_val = float(edt.max())

        if peak_val == 0.0:
            pt_crop = _nearest_valid_pixel(frag_crop, (0, 0))
            pt      = (pt_crop[0] + c0, pt_crop[1] + r0) if pt_crop else None
            score   = 0.0
        else:
            threshold = peak_val * 0.99
            plateau   = (edt >= threshold) & frag_crop
            ys, xs    = np.where(plateau)
            cx, cy    = float(xs.mean()), float(ys.mean())   # crop-local float
            # Round in full-image space (round-half-to-even is not invariant
            # under integer offsets); map back to a crop-local index for the
            # EDT lookup.
            fx, fy    = int(round(cx + c0)), int(round(cy + r0))
            pt        = (fx, fy)
            pt_crop   = (fx - c0, fy - r0)
            score     = _score_at_point(edt, pt_crop)

        results.append({
            "point": pt, "score": round(score, 2), "area_px2": area,
            "is_main": (i == 0), "valid": pt is not None, "algorithm": "EDT_PEAK",
        })
    return results


# --- Dispatch ---
SAMPLER_FN = {
    "CENTROID": sample_centroid,
    "EDT_PEAK": sample_edt_peak,
}

def run_samplers(valid_mask: np.ndarray, params: dict, algorithms: list) -> dict:
    return {
        alg: SAMPLER_FN[alg](valid_mask, params)
        for alg in algorithms if alg in SAMPLER_FN
    }

print("Module 3 (samplers) ready.")

In [ ]:
# === Cell 9: Interactive Sampling Viewer ===

# --- Style config (16 cm LaTeX text width) ---
_FW        = 16 / 2.54
_FONT_SIZE = 8

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE
plt.rcParams["axes.titlesize"]  = _FONT_SIZE
plt.rcParams["axes.labelsize"]  = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150

# --- Visual style ---
CLASS_COLORS_RGB = {
    0: (255,  0,  0),   # body  — red
    1: ( 0, 255,  0),   # cap   — green
    2: ( 0, 0, 255),   # label — blue
}
# Normalised Matplotlib colours per class
CLASS_COLORS_MPL = {
    cid: tuple(c / 255 for c in rgb)
    for cid, rgb in CLASS_COLORS_RGB.items()
}
MASK_ALPHA = 0.30

# Algorithm markers
ALG_MARKERS = {
    "CENTROID": "+",
    "EDT_PEAK": "x",
}

# Valid-body overlay colour and opacity
VALID_BODY_COLOR = np.array([255, 60, 60], dtype=float)
VALID_BODY_ALPHA = 0.35


# --- Helpers ---
def _get_image(stem: str, show_bg: bool):
    """Return the source image as RGB, or a white canvas."""
    if show_bg and IMAGE_DIR is not None:
        for ext in (".jpg", ".jpeg", ".png", ".bmp"):
            p = IMAGE_DIR / (stem + ext)
            if p.exists():
                bgr = cv2.imread(str(p))
                if bgr is not None:
                    return cv2.cvtColor(
                        cv2.resize(bgr, (IMG_W, IMG_H)), cv2.COLOR_BGR2RGB
                    )
    return np.full((IMG_H, IMG_W, 3), 255, dtype=np.uint8)


def _get_labels(stem: str, source: str) -> list:
    """Retrieve instances for a stem from the selected source."""
    if source == "Predicted":
        return LABELS_PRED.get(stem, [])
    return LABELS_GT.get(stem, [])


def render_frame(
    stem, source, sensor_radius, min_area, active_algs,
    show_masks, show_valid_body, show_contours, show_bg,
):
    """Build a matplotlib figure and return (fig, timing_ms)."""
    instances = _get_labels(stem, source)
    mask_data = build_masks(instances)
    params    = {"sensor_radius": sensor_radius, "min_area": min_area}

    canvas = _get_image(stem, show_bg).copy().astype(float)

    # --- Class masks ---
    if show_masks:
        overlay = canvas.copy()
        for cls_name in ("body", "cap", "label"):
            cls_id = CLASS_MAP[cls_name]
            rgb    = CLASS_COLORS_RGB[cls_id]
            for entry in mask_data[cls_name]:
                ys, xs = np.where(entry["raw"])
                overlay[ys, xs] = rgb
        canvas = canvas * (1 - MASK_ALPHA) + overlay * MASK_ALPHA

    # --- Valid-body overlay ---
    if show_valid_body:
        for entry in mask_data["body"]:
            ys, xs = np.where(entry["valid"])
            canvas[ys, xs] = (
                canvas[ys, xs] * (1 - VALID_BODY_ALPHA)
                + VALID_BODY_COLOR * VALID_BODY_ALPHA
            )

    fig, ax = plt.subplots(figsize=(_FW, _FW * 0.58))
    ax.imshow(canvas.astype(np.uint8))

    # --- Polygon contours ---
    if show_contours:
        for cls_name in ("body", "cap", "label"):
            cls_id = CLASS_MAP[cls_name]
            mpl_c  = CLASS_COLORS_MPL[cls_id]
            for entry in mask_data[cls_name]:
                poly = entry["polygon_px"]
                if len(poly) < 3:
                    continue
                xs_p = [p[0] for p in poly] + [poly[0][0]]
                ys_p = [p[1] for p in poly] + [poly[0][1]]
                ax.plot(xs_p, ys_p, color=mpl_c, lw=1.2, alpha=0.8)

    # --- Timed sampling ---
    t0 = time.perf_counter()
    all_sampling = {}  # (cls_name, inst_idx) -> {alg: results}
    for cls_name in ("body", "cap", "label"):
        for inst_idx, entry in enumerate(mask_data[cls_name]):
            sampling = run_samplers(entry["valid"], params, active_algs)
            all_sampling[(cls_name, inst_idx)] = sampling
    t1 = time.perf_counter()
    timing_ms = (t1 - t0) * 1000.0

    # --- Sampled points ---
    for (cls_name, inst_idx), sampling in all_sampling.items():
        cls_id = CLASS_MAP[cls_name]
        mpl_c  = CLASS_COLORS_MPL[cls_id]
        bright = tuple(max(0.0, c * 0.4) for c in mpl_c)

        for alg, res_list in sampling.items():
            marker = ALG_MARKERS.get(alg, "o")
            for res in res_list:
                if not res["valid"] or res["point"] is None:
                    continue
                px, py = res["point"]
                size   = 220 if res["is_main"] else 90
                lw     = 3.0 if res["is_main"] else 1.8
                zorder = 6   if res["is_main"] else 5

                ax.scatter(
                    px, py, marker=marker, s=size,
                    c=[bright], linewidths=lw,
                    edgecolors="white" if res["is_main"] else "none",
                    zorder=zorder,
                )
                # Sensor footprint circle on main points
                if res["is_main"]:
                    ax.add_patch(plt.Circle(
                        (px, py), sensor_radius,
                        color=bright, fill=False, lw=1.0, alpha=0.6, zorder=5,
                    ))
                # Label: algorithm, class/instance, main marker (*), score
                tag = (
                    f"{alg[:3]} "
                    f"{cls_name[0].upper()}{inst_idx}"
                    f"{'*' if res['is_main'] else ''}"
                    f" s={res['score']:.0f}px"
                )
                ax.annotate(
                    tag, (px, py),
                    xytext=(6, -10 if alg == "EDT_PEAK" else 6),
                    textcoords="offset points",
                    fontsize=_FONT_SIZE - 3, color=bright, zorder=7,
                )

    # --- Legend ---
    handles = [
        mpatches.Patch(color=CLASS_COLORS_MPL[CLASS_MAP[n]], label=n)
        for n in CLASS_MAP
    ]
    for alg in active_algs:
        handles.append(plt.Line2D(
            [], [], marker=ALG_MARKERS.get(alg, "o"),
            color="black", markerfacecolor="black", markeredgecolor="black",
            markersize=8, linestyle="None", label=alg,
        ))
    ax.legend(handles=handles, loc="center right", fontsize=_FONT_SIZE - 1,
              framealpha=0.7)

    ax.axis("off")
    plt.tight_layout()
    return fig, timing_ms


# --- Widget ---
if not ALL_STEMS:
    print("No label files found — check label paths in the config cell.")
else:
    file_dd = widgets.Dropdown(
        options=[(s, s) for s in ALL_STEMS],
        value=ALL_STEMS[0],
        description="File:",
        layout=widgets.Layout(width="700px"),
    )
    source_toggle = widgets.ToggleButtons(
        options=["Predicted", "GT"],
        value="Predicted",
        description="Source:",
        style={"button_width": "100px"},
    )
    alg_cbs = [
        widgets.Checkbox(value=True, description=alg, indent=False)
        for alg in ALGORITHMS
    ]
    radius_slider = widgets.IntSlider(
        value=SENSOR_RADIUS_PX, min=0, max=80, step=1,
        description="Sensor r [px]:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="380px"),
    )
    minarea_slider = widgets.IntSlider(
        value=MIN_VALID_AREA_PX2, min=0, max=5000, step=50,
        description="Min area [px²]:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="380px"),
    )
    masks_cb      = widgets.Checkbox(value=True,  description="Masks",      indent=False)
    valid_body_cb = widgets.Checkbox(value=True,  description="Valid body", indent=False)
    contours_cb   = widgets.Checkbox(value=True,  description="Contours",   indent=False)
    bg_cb         = widgets.Checkbox(value=True,  description="Background", indent=False)

    random_btn = widgets.Button(description="Random",   button_style="info",    icon="random")
    refresh_btn = widgets.Button(description="Refresh", button_style="primary", icon="refresh")
    save_btn   = widgets.Button(description="Save PNG",                          icon="save")

    viewer_out = widgets.Output()

    def _render_w(save_path=None):
        stem        = file_dd.value
        source      = source_toggle.value
        sensor_r    = radius_slider.value
        min_area    = minarea_slider.value
        active_algs = [cb.description for cb in alg_cbs if cb.value]
        with viewer_out:
            clear_output(wait=True)
            if not active_algs:
                print("No algorithms selected.")
                return
            fig, t_ms = render_frame(
                stem, source, sensor_r, min_area, active_algs,
                masks_cb.value, valid_body_cb.value,
                contours_cb.value, bg_cb.value,
            )
            if save_path:
                fig.savefig(save_path, bbox_inches="tight", dpi=150)
                print(f"Saved: {save_path}")
            plt.show()

    # Re-render on file or source change
    file_dd.observe(lambda c: _render_w() if c["name"] == "value" else None)
    source_toggle.observe(lambda c: _render_w() if c["name"] == "value" else None)

    random_btn.on_click(lambda b: (
        setattr(file_dd, "value", random.choice(ALL_STEMS)),
    ))
    refresh_btn.on_click(lambda b: _render_w())
    def _on_save(b):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        sp = RESULTS_DIR / f"{PROJECT_NAME}_widget_{file_dd.value}_{ts}.png"
        _render_w(save_path=sp)
    save_btn.on_click(_on_save)

    display(
        widgets.VBox([
            widgets.HBox([file_dd, source_toggle]),
            widgets.HBox([radius_slider, minarea_slider]),
            widgets.HBox([
                widgets.Label("Algorithms:"), *alg_cbs,
                widgets.Label("  Layers:"),
                masks_cb, valid_body_cb, contours_cb, bg_cb,
            ]),
            widgets.HBox([refresh_btn, random_btn, save_btn]),
        ]),
        viewer_out,
    )

In [ ]:
# === Cell 10: Batch Runner and CSV Export ===

def run_batch(
    all_labels: dict,
    algorithms: list,
    source_name: str,
    sensor_radius: int = SENSOR_RADIUS_PX,
    min_area: int      = MIN_VALID_AREA_PX2,
) -> pd.DataFrame:
    params  = {"sensor_radius": sensor_radius, "min_area": min_area}
    records = []

    for stem, instances in all_labels.items():
        mask_data = build_masks(instances)

        for cls_name, cls_id in CLASS_MAP.items():
            for inst_idx, entry in enumerate(mask_data[cls_name]):
                valid_mask = entry["valid"]

                # Each algorithm is timed per instance; the time is written to
                # every fragment row, and aggregation uses only the is_main row
                # (one per instance and algorithm).
                for alg in algorithms:
                    if alg not in SAMPLER_FN:
                        continue
                    t0       = time.perf_counter()
                    res_list = SAMPLER_FN[alg](valid_mask, params)
                    dt_ms    = (time.perf_counter() - t0) * 1000.0

                    for frag_idx, res in enumerate(res_list):
                        pt = res["point"]
                        records.append({
                            "file":           stem,
                            "source":         source_name,
                            "class_id":       cls_id,
                            "class_name":     cls_name,
                            "instance_idx":   inst_idx,
                            "algorithm":      alg,
                            "fragment_idx":   frag_idx,
                            "is_main":        res["is_main"],
                            "point_x":        pt[0] if pt else None,
                            "point_y":        pt[1] if pt else None,
                            "valid_area_px2": res["area_px2"],
                            "score":          res["score"],
                            "valid":          res["valid"],
                            "time_ms":        round(dt_ms, 4),
                        })
    return pd.DataFrame(records)


# Run on predicted labels
print("Running batch on predicted labels...")
t0 = time.perf_counter()
df_pred = run_batch(LABELS_PRED, ALGORITHMS, "predicted")
t_pred  = time.perf_counter() - t0
print(f"  {len(df_pred)} rows in {t_pred:.1f}s")

# Run on GT labels
print("Running batch on GT labels...")
t0 = time.perf_counter()
df_gt = run_batch(LABELS_GT, ALGORITHMS, "gt")
t_gt  = time.perf_counter() - t0
print(f"  {len(df_gt)} rows in {t_gt:.1f}s")

# Merge and save the detail CSV
batch_df = pd.concat([df_pred, df_gt], ignore_index=True)
csv_path = RESULTS_DIR / f"{PROJECT_NAME}_sampling_results.csv"
batch_df.to_csv(csv_path, index=False)

# --- Aggregated summary: yield, score statistics, throughput ---
# Main points only (one representative point per instance).
main_df = batch_df[batch_df["is_main"]].copy()

# Yield: share of instances with a valid main point
yield_stats = (
    main_df
    .groupby(["source", "algorithm", "class_name"])["valid"]
    .agg(n_total="count", n_valid="sum")
)
yield_stats["yield_pct"] = yield_stats["n_valid"] / yield_stats["n_total"] * 100

# Score statistics [px] over valid main points only
score_stats = (
    main_df[main_df["valid"]]
    .groupby(["source", "algorithm", "class_name"])["score"]
    .agg(
        score_mean   = "mean",
        score_std    = "std",
        score_median = "median",
        score_min    = "min",
        score_max    = "max",
    )
)

# Throughput [points/s] per instance = 1000 / per-instance time_ms.
main_df["throughput_pts_s"] = 1000.0 / main_df["time_ms"]
speed_stats = (
    main_df
    .groupby(["source", "algorithm", "class_name"])["throughput_pts_s"]
    .agg(thrpt_mean="mean", thrpt_std="std")
)

agg_summary = (
    yield_stats
    .join(score_stats)
    .join(speed_stats)
    .round(1)
)
agg_summary = agg_summary[[
    "n_total", "n_valid", "yield_pct",
    "score_mean", "score_std", "score_median", "score_min", "score_max",
    "thrpt_mean", "thrpt_std",
]]

# Save the aggregated summary CSV
agg_csv_path = RESULTS_DIR / f"{PROJECT_NAME}_sampling_summary.csv"
agg_summary.to_csv(agg_csv_path)

print(f"Saved detail CSV:   {csv_path}")
print(f"Saved summary CSV:  {agg_csv_path}")
print()
print("Main-point yield, score statistics [px], and throughput [points/s]")
print("per source / algorithm / class:")
display(agg_summary)

In [ ]:
# === Cell 11: Performance Assessment Figure ===
# 3-panel figure from the sampling results CSV (main points only): (a) clearance
# per region and algorithm, (b) body clearance histogram, (c) GT vs predicted
# median clearance. Saved as a boxplot and a violin version.

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import pandas as pd
import numpy as np
from pathlib import Path

# --- Figure size and font ---
_FW        = 16 / 2.54          # reference = LaTeX \textwidth in inches
_FIG_W     = _FW * 1.5          # overall figure width
_FONT_SIZE = 8                   # base font size

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE        # body text, tick labels
plt.rcParams["axes.titlesize"]  = _FONT_SIZE        # panel subtitles
plt.rcParams["axes.labelsize"]  = _FONT_SIZE        # x/y axis labels
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1    # legend entries
plt.rcParams["savefig.dpi"]     = 150

# --- Paths and constants ---
RESULTS_CSV  = RESULTS_DIR / f"{PROJECT_NAME}_sampling_results.csv"
FIG_STEM     = RESULTS_DIR / f"{PROJECT_NAME}_performance_assessment"
REGIONS      = ["body", "cap", "label"]
ALGS         = ["CENTROID", "EDT_PEAK"]
FAIL_PX      = 5
SENSOR_R_PX  = globals().get("SENSOR_RADIUS_PX", 44)
C_CEN, C_EDT = "#d1495b", "#2a9d8f"   # algorithm colours shared across all panels

# --- Helpers ---
def _panel_label(ax, letter, title):
    """Bold (a)/(b)/(c) prefix in front of the panel title."""
    ax.set_title(f"$\\bf{{({letter})}}$ {title}")

def _subplot_c(ax, med_df, panel_letter="c"):
    """Panel (c): algorithm colours; solid = predicted, hatched = GT."""
    alg_colors = [C_CEN if a == "CENTROID" else C_EDT for _, a in med_df.index]
    xlabels    = [f"{r}\n{a[:3]}" for r, a in med_df.index]
    x = np.arange(len(med_df))
    w = 0.35
    ax.bar(x - w/2, med_df["gt"], width=w, color=alg_colors,
           alpha=0.65, hatch="///", edgecolor="white", linewidth=0.5, zorder=2)
    ax.bar(x + w/2, med_df["predicted"], width=w, color=alg_colors,
           alpha=0.85, edgecolor="black", linewidth=0.3, zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=_FONT_SIZE - 2)
    ax.set_ylabel("Median Clearance Radius (px)")
    ax.grid(axis="y", alpha=0.25)
    ax.set_box_aspect(1)
    _panel_label(ax, panel_letter, "Predicted vs. Ground Truth Median Clearance")
    ax.legend(handles=[
        Patch(facecolor=C_CEN, alpha=0.85, edgecolor="black", lw=0.3, label="CENTROID"),
        Patch(facecolor=C_EDT, alpha=0.85, edgecolor="black", lw=0.3, label="EDT_PEAK"),
        Patch(facecolor="grey", alpha=0.85, edgecolor="black", lw=0.3, label="Predicted"),
        Patch(facecolor="grey", alpha=0.65, hatch="///", edgecolor="white",
              lw=0.5, label="GT"),
    ], fontsize=_FONT_SIZE - 2, ncols=2)

# --- Load ---
df   = pd.read_csv(RESULTS_CSV)
main = df[df["is_main"]].copy()

# --- Figure 1: boxplot version ---
fig, ax = plt.subplots(1, 3, figsize=(_FIG_W, _FIG_W / 3))

# --- (a) Clearance by region and algorithm (boxplot) ---
data, positions, colors, tickpos = [], [], [], []
pos = 0
for r in REGIONS:
    for a in ALGS:
        sub = main[(main["source"] == "predicted") &
                   (main["class_name"] == r) &
                   (main["algorithm"] == a)]["score"].values
        data.append(sub); positions.append(pos)
        colors.append(C_CEN if a == "CENTROID" else C_EDT); pos += 1
    tickpos.append(pos - 1.5); pos += 0.8

bp = ax[0].boxplot(data, positions=positions, widths=0.7, patch_artist=True,
                   showfliers=False, showmeans=True, meanline=True,
                   medianprops=dict(color="black", lw=1.3, linestyle="-"),
                   meanprops=dict(color="black",   lw=1.1, linestyle="--"))
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c); patch.set_alpha(0.78)

ax[0].axhline(SENSOR_R_PX, ls=":", c="#888", lw=1)
ax[0].text(positions[-1], SENSOR_R_PX + 2, f"sensor r = {SENSOR_R_PX} px",
           fontsize=_FONT_SIZE - 2, ha="right", color="#666")
ax[0].set_xticks(tickpos)
ax[0].set_xticklabels(REGIONS)
ax[0].set_ylabel("Clearance Radius (px)")
ax[0].set_box_aspect(1)
_panel_label(ax[0], "a", "Clearance Radius by Region × Algorithm")
ax[0].legend(handles=[
    Patch(facecolor=C_CEN, alpha=.78, label="CENTROID"),
    Patch(facecolor=C_EDT, alpha=.78, label="EDT_PEAK"),
    Line2D([0], [0], color="black", lw=1.3, linestyle="-",  label="Median"),
    Line2D([0], [0], color="black", lw=1.1, linestyle="--", label="Mean"),
], fontsize=_FONT_SIZE - 2, ncols=2)

# --- (b) Body clearance histogram ---
b_cen = main[(main["class_name"] == "body") & (main["algorithm"] == "CENTROID")]["score"]
b_edt = main[(main["class_name"] == "body") & (main["algorithm"] == "EDT_PEAK")]["score"]
hi    = max(b_cen.max(), b_edt.max())
bins  = np.linspace(0, hi, 41)
ax[1].hist(b_cen, bins=bins, color=C_CEN, alpha=.70, label="CENTROID")
ax[1].hist(b_edt, bins=bins, color=C_EDT, alpha=.55, label="EDT_PEAK")
ax[1].axvspan(0, FAIL_PX, color="red", alpha=.12)
pct_cen = 100 * (b_cen <= FAIL_PX).mean()
pct_edt = 100 * (b_edt <= FAIL_PX).mean()
ax[1].text(hi * 0.03, ax[1].get_ylim()[1] * 0.88,
           f"\u2264 {FAIL_PX} px:\nCEN {pct_cen:.0f}% vs EDT {pct_edt:.0f}%",
           fontsize=_FONT_SIZE - 2, color="#a00")
ax[1].set_xlabel("Clearance Radius (px)")
ax[1].set_ylabel("Body Instances")
ax[1].set_box_aspect(1)
_panel_label(ax[1], "b", "Body Region Clearance Radius")
ax[1].legend(fontsize=_FONT_SIZE - 1)

# --- (c) GT vs predicted median clearance ---
med = (main.groupby(["class_name", "algorithm", "source"])["score"]
           .median().unstack("source"))
med = med.reindex(pd.MultiIndex.from_product([REGIONS, ALGS],
                  names=["class_name", "algorithm"]))
_subplot_c(ax[2], med, panel_letter="c")

fig.tight_layout()
fig.savefig(f"{FIG_STEM}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_STEM}.png")


# --- Figure 2: violin version (panel (a) differs) ---
fig2, ax2 = plt.subplots(1, 3, figsize=(_FIG_W, _FIG_W / 3))

# --- (a) Clearance by region and algorithm (violin) ---
v_data, v_pos, v_colors, v_tickpos = [], [], [], []
pos = 0
for r in REGIONS:
    grp_pos = []
    for a in ALGS:
        sub = main[(main["source"] == "predicted") &
                   (main["class_name"] == r) &
                   (main["algorithm"] == a)]["score"].values
        if sub.size:
            v_data.append(sub)
            v_pos.append(pos)
            v_colors.append(C_CEN if a == "CENTROID" else C_EDT)
        grp_pos.append(pos)
        pos += 1
    v_tickpos.append(float(np.mean(grp_pos)))
    pos += 0.8

if v_data:
    vp = ax2[0].violinplot(v_data, positions=v_pos, widths=0.7,
                           showmeans=True, showmedians=True, showextrema=False)
    for body, c in zip(vp["bodies"], v_colors):
        body.set_facecolor(c); body.set_alpha(0.78)
        body.set_edgecolor("#333"); body.set_linewidth(0.6)
    vp["cmedians"].set_color("black"); vp["cmedians"].set_linewidth(1.3)
    vp["cmeans"].set_color("black");   vp["cmeans"].set_linestyle("--")
    vp["cmeans"].set_linewidth(1.1)

ax2[0].axhline(SENSOR_R_PX, ls=":", c="#888", lw=1)
ax2[0].text(v_pos[-1] if v_pos else 0, SENSOR_R_PX + 2,
            f"sensor r = {SENSOR_R_PX} px",
            fontsize=_FONT_SIZE - 2, ha="right", color="#666")
ax2[0].set_xticks(v_tickpos)
ax2[0].set_xticklabels(REGIONS)
ax2[0].set_ylabel("Clearance Radius (px)")
ax2[0].set_box_aspect(1)
_panel_label(ax2[0], "a", "Clearance Radius by Region × Algorithm")
ax2[0].legend(handles=[
    Patch(facecolor=C_CEN, alpha=.78, label="CENTROID"),
    Patch(facecolor=C_EDT, alpha=.78, label="EDT_PEAK"),
    Line2D([0], [0], color="black", lw=1.3, linestyle="-",  label="Median"),
    Line2D([0], [0], color="black", lw=1.1, linestyle="--", label="Mean"),
], fontsize=_FONT_SIZE - 2, ncols=2)

# --- (b) Body clearance histogram ---
b_cen = main[(main["class_name"] == "body") & (main["algorithm"] == "CENTROID")]["score"]
b_edt = main[(main["class_name"] == "body") & (main["algorithm"] == "EDT_PEAK")]["score"]
hi    = max(b_cen.max(), b_edt.max())
bins  = np.linspace(0, hi, 41)
ax2[1].hist(b_cen, bins=bins, color=C_CEN, alpha=.70, label="CENTROID")
ax2[1].hist(b_edt, bins=bins, color=C_EDT, alpha=.55, label="EDT_PEAK")
ax2[1].axvspan(0, FAIL_PX, color="red", alpha=.12)
pct_cen = 100 * (b_cen <= FAIL_PX).mean()
pct_edt = 100 * (b_edt <= FAIL_PX).mean()
ax2[1].text(hi * 0.03, ax2[1].get_ylim()[1] * 0.88,
            f"\u2264 {FAIL_PX} px:\nCEN {pct_cen:.0f}% vs EDT {pct_edt:.0f}%",
            fontsize=_FONT_SIZE - 2, color="#a00")
ax2[1].set_xlabel("Clearance Radius (px)")
ax2[1].set_ylabel("Body Instances")
ax2[1].set_box_aspect(1)
_panel_label(ax2[1], "b", "Body Region Clearance Radius")
ax2[1].legend(fontsize=_FONT_SIZE - 1)

# --- (c) GT vs predicted median clearance ---
med_v = (main.groupby(["class_name", "algorithm", "source"])["score"]
             .median().unstack("source"))
med_v = med_v.reindex(pd.MultiIndex.from_product([REGIONS, ALGS],
                      names=["class_name", "algorithm"]))
_subplot_c(ax2[2], med_v, panel_letter="c")

fig2.tight_layout()
fig2.savefig(f"{FIG_STEM}_violin.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_STEM}_violin.png")

In [ ]:
# === Cell 12: Line Scanner Grid Demo ===

# --- Style config (16 cm LaTeX text width) ---
_FW        = 16 / 2.54
_FONT_SIZE = 8

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE
plt.rcParams["axes.titlesize"]  = _FONT_SIZE
plt.rcParams["axes.labelsize"]  = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150

# Millimetre-to-pixel calibration from the config px/cm ratio
PIXELS_PER_MM = PIXELS_PER_CM / 10.0   # e.g. 35.5 px/cm -> 3.55 px/mm


def grid_scan(
    mask_data: dict,
    spacing_x: int,
    spacing_y: int,
    erosion_px: int = 0,
    img_w: int = IMG_W,
    img_h: int = IMG_H,
) -> dict:
    """
    Classify every grid point against the valid instance masks.

    Returns
    -------
    dict with keys:
        'grid_xs', 'grid_ys' : 1-D arrays of grid coordinates
        'hits'   : list of (x, y, cls_name, inst_idx)  — valid single-mask hits
        'misses' : list of (x, y)                      — no-mask or multi-mask
        'counts' : dict  (cls_name, inst_idx) -> int   — hit count per instance
    """
    grid_xs = np.arange(0, img_w, spacing_x)
    grid_ys = np.arange(0, img_h, spacing_y)

    mask_entries = []   # (cls_name, inst_idx, test_mask)
    for cls_name in ("body", "cap", "label"):
        for inst_idx, entry in enumerate(mask_data[cls_name]):
            vmask = entry["valid"]
            if erosion_px > 0:
                test_mask = distance_transform_edt(vmask) >= erosion_px
            else:
                test_mask = vmask
            mask_entries.append((cls_name, inst_idx, test_mask))

    hits    = []
    misses  = []
    counts  = defaultdict(int)

    for y in grid_ys:
        for x in grid_xs:
            matched = []
            for cls_name, inst_idx, vmask in mask_entries:
                if vmask[y, x]:
                    matched.append((cls_name, inst_idx))
            if len(matched) == 1:
                cn, ii = matched[0]
                hits.append((int(x), int(y), cn, ii))
                counts[(cn, ii)] += 1
            else:
                misses.append((int(x), int(y)))

    return {
        "grid_xs": grid_xs,
        "grid_ys": grid_ys,
        "hits":    hits,
        "misses":  misses,
        "counts":  dict(counts),
    }


def grid_hits_dataframe(scan: dict, stem: str, source: str,
                        spacing_x_mm: float, spacing_y_mm: float,
                        erosion_mm: float) -> pd.DataFrame:
    """
    Per-region measurement-point coordinates of a grid scan, in pixels and
    millimetres.
    """
    CLASS_ORDER = {"body": 0, "cap": 1, "label": 2}
    ordered = sorted(
        scan["hits"],
        key=lambda h: (CLASS_ORDER.get(h[2], 9), h[3], h[1], h[0]),
    )
    rows = []
    per_inst_idx = defaultdict(int)
    for x, y, cn, ii in ordered:
        rows.append({
            "stem":          stem,
            "source":        source,
            "class_name":    cn,
            "instance_idx":  ii,
            "point_idx":     per_inst_idx[(cn, ii)],
            "point_x_px":    x,
            "point_y_px":    y,
            "point_x_mm":    round(x / PIXELS_PER_MM, 2),
            "point_y_mm":    round(y / PIXELS_PER_MM, 2),
            "spacing_x_mm":  spacing_x_mm,
            "spacing_y_mm":  spacing_y_mm,
            "erosion_mm":    erosion_mm,
        })
        per_inst_idx[(cn, ii)] += 1
    return pd.DataFrame(rows)


def render_grid_frame(
    stem, source, spacing_x_mm, spacing_y_mm, erosion_mm,
    show_masks, show_contours, show_bg,
):
    """Build a matplotlib figure for the grid/line-scanner demo (mm inputs)."""
    instances = _get_labels(stem, source)
    mask_data = build_masks(instances)

    # Convert millimetre controls to pixel space for the scan
    spacing_x = max(1, int(round(spacing_x_mm * PIXELS_PER_MM)))
    spacing_y = max(1, int(round(spacing_y_mm * PIXELS_PER_MM)))
    erosion_px = int(round(erosion_mm * PIXELS_PER_MM))

    t0 = time.perf_counter()
    scan = grid_scan(mask_data, spacing_x, spacing_y, erosion_px)
    t1 = time.perf_counter()
    timing_ms = (t1 - t0) * 1000.0

    canvas = _get_image(stem, show_bg).copy().astype(float)

    # --- Class masks ---
    if show_masks:
        overlay = canvas.copy()
        for cls_name in ("body", "cap", "label"):
            cls_id = CLASS_MAP[cls_name]
            rgb    = CLASS_COLORS_RGB[cls_id]
            for entry in mask_data[cls_name]:
                ys, xs = np.where(entry["raw"])
                overlay[ys, xs] = rgb
        canvas = canvas * (1 - MASK_ALPHA) + overlay * MASK_ALPHA

    fig, ax = plt.subplots(figsize=(_FW, _FW * 0.58))
    ax.imshow(canvas.astype(np.uint8))

    # --- Polygon contours ---
    if show_contours:
        for cls_name in ("body", "cap", "label"):
            cls_id = CLASS_MAP[cls_name]
            mpl_c  = CLASS_COLORS_MPL[cls_id]
            for entry in mask_data[cls_name]:
                poly = entry["polygon_px"]
                if len(poly) < 3:
                    continue
                xs_p = [p[0] for p in poly] + [poly[0][0]]
                ys_p = [p[1] for p in poly] + [poly[0][1]]
                ax.plot(xs_p, ys_p, color=mpl_c, lw=1.2, alpha=0.8)

    # --- Scan lines ---
    for y in scan["grid_ys"]:
        ax.axhline(y=y, color="0.65", lw=0.4, zorder=2)

    # --- Miss points ---
    if scan["misses"]:
        mx = [p[0] for p in scan["misses"]]
        my = [p[1] for p in scan["misses"]]
        ax.scatter(mx, my, s=2, c="0.55", marker=".", zorder=3, linewidths=0)

    # --- Hit points (class-coloured) ---
    hits_by_class = defaultdict(lambda: ([], []))
    for x, y, cn, ii in scan["hits"]:
        cls_id = CLASS_MAP[cn]
        hits_by_class[cls_id][0].append(x)
        hits_by_class[cls_id][1].append(y)

    for cls_id, (hx, hy) in hits_by_class.items():
        mpl_c  = CLASS_COLORS_MPL[cls_id]
        bright = tuple(min(1.0, c * 1.4 + 0.15) for c in mpl_c)
        ax.scatter(hx, hy, s=3, c=[bright], marker="o", zorder=4, linewidths=0)

    # --- Legend ---
    handles = [
        mpatches.Patch(color=CLASS_COLORS_MPL[CLASS_MAP[n]], label=n)
        for n in CLASS_MAP
    ]
    ax.legend(handles=handles, loc="center right", fontsize=_FONT_SIZE - 1,
              framealpha=0.7)

    ax.axis("off")
    plt.tight_layout()
    return fig, timing_ms, scan


# --- Widget ---
if not ALL_STEMS:
    print("No label files found.")
else:
    # Default slider values [mm]
    _def_sx_mm = 3.5
    _def_sy_mm = 3.5
    _def_er_mm = 1.0

    # Holds the most recently rendered scan for CSV export
    _grid_state = {}

    g_file_dd = widgets.Dropdown(
        options=[(s, s) for s in ALL_STEMS],
        value=ALL_STEMS[0],
        description="File:",
        layout=widgets.Layout(width="700px"),
    )
    g_source_toggle = widgets.ToggleButtons(
        options=["Predicted", "GT"],
        value="Predicted",
        description="Source:",
        style={"button_width": "100px"},
    )
    g_sx_slider = widgets.FloatSlider(
        value=_def_sx_mm, min=1.0, max=30.0, step=0.5,
        description="X spacing [mm]:", readout_format=".1f",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="380px"),
    )
    g_sy_slider = widgets.FloatSlider(
        value=_def_sy_mm, min=1.0, max=30.0, step=0.5,
        description="Y spacing [mm]:", readout_format=".1f",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="380px"),
    )
    g_erosion_slider = widgets.FloatSlider(
        value=_def_er_mm, min=0.0, max=25.0, step=0.5,
        description="Erosion [mm]:", readout_format=".1f",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="380px"),
    )
    g_masks_cb    = widgets.Checkbox(value=True,  description="Masks",      indent=False)
    g_contours_cb = widgets.Checkbox(value=True,  description="Contours",   indent=False)
    g_bg_cb       = widgets.Checkbox(value=True,  description="Background", indent=False)

    g_refresh_btn = widgets.Button(description="Refresh", button_style="primary", icon="refresh")
    g_random_btn  = widgets.Button(description="Random",  button_style="info",    icon="random")
    g_save_btn    = widgets.Button(description="Save PNG",                         icon="save")
    g_savecsv_btn = widgets.Button(description="Save CSV", button_style="success", icon="table")

    g_viewer_out = widgets.Output()

    def _render_g(save_path=None):
        stem   = g_file_dd.value
        source = g_source_toggle.value
        sx     = g_sx_slider.value
        sy     = g_sy_slider.value
        er     = g_erosion_slider.value
        with g_viewer_out:
            clear_output(wait=True)
            fig, t_ms, scan = render_grid_frame(
                stem, source, sx, sy, er,
                g_masks_cb.value, g_contours_cb.value, g_bg_cb.value,
            )
            # Keep the latest scan for CSV export
            _grid_state.update(
                scan=scan, stem=stem, source=source,
                sx_mm=sx, sy_mm=sy, er_mm=er,
            )
            if save_path:
                fig.savefig(save_path, bbox_inches="tight", dpi=150)
                print(f"Saved: {save_path}")
            plt.show()

    g_file_dd.observe(lambda c: _render_g() if c["name"] == "value" else None)
    g_source_toggle.observe(lambda c: _render_g() if c["name"] == "value" else None)
    g_sx_slider.observe(lambda c: _render_g() if c["name"] == "value" else None)
    g_sy_slider.observe(lambda c: _render_g() if c["name"] == "value" else None)
    g_erosion_slider.observe(lambda c: _render_g() if c["name"] == "value" else None)

    g_refresh_btn.on_click(lambda b: _render_g())
    g_random_btn.on_click(lambda b: (
        setattr(g_file_dd, "value", random.choice(ALL_STEMS)),
    ))

    def _on_save_g(b):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        sp = RESULTS_DIR / f"{PROJECT_NAME}_grid_{g_file_dd.value}_{ts}.png"
        _render_g(save_path=sp)
    g_save_btn.on_click(_on_save_g)

    def _on_save_csv(b):
        with g_viewer_out:
            if not _grid_state.get("scan"):
                print("Nothing to export yet — render a frame first.")
                return
            df = grid_hits_dataframe(
                _grid_state["scan"], _grid_state["stem"], _grid_state["source"],
                _grid_state["sx_mm"], _grid_state["sy_mm"], _grid_state["er_mm"],
            )
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            sp = RESULTS_DIR / f"{PROJECT_NAME}_grid_points_{_grid_state['stem']}_{ts}.csv"
            df.to_csv(sp, index=False)
            print(f"Saved {len(df)} measurement points to: {sp}")
    g_savecsv_btn.on_click(_on_save_csv)

    display(
        widgets.VBox([
            widgets.HBox([g_file_dd, g_source_toggle]),
            widgets.HBox([g_sx_slider, g_sy_slider]),
            widgets.HBox([g_erosion_slider]),
            widgets.HBox([
                widgets.Label("Layers:"),
                g_masks_cb, g_contours_cb, g_bg_cb,
            ]),
            widgets.HBox([g_refresh_btn, g_random_btn, g_save_btn, g_savecsv_btn]),
        ]),
        g_viewer_out,
    )

In [ ]:
# === Cell 13: Grid Point-Count Statistics ===
# Runs grid_scan (Line Scanner Grid Demo cell) over all stems of both label sources,
# saves per-image per-region point counts to CSV, prints statistics and saves
# GT-vs-predicted boxplot and split-violin figures.

import matplotlib.patheffects
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np

# --- Figure size and font ---
_FW        = 16 / 2.54      # reference = LaTeX \textwidth in inches
_FONT_SIZE = 10              # base font size

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE        # body text, tick labels
plt.rcParams["axes.titlesize"]  = _FONT_SIZE        # panel titles
plt.rcParams["axes.labelsize"]  = _FONT_SIZE        # x/y axis labels
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1    # legend entries
plt.rcParams["savefig.dpi"]     = 150

# Region colours
C_BODY  = "#FF0000"
C_CAP   = "#00FF00"
C_LABEL = "#0000FF"
region_color = {"body": C_BODY, "cap": C_CAP, "label": C_LABEL}

# --- Batch grid parameters ---
BATCH_SPACING_X_MM = 3.5    # matches the representative object-level GSD
BATCH_SPACING_Y_MM = 3.5
BATCH_EROSION_MM   = 1.0


def batch_grid_pointcounts(spacing_x_px: int, spacing_y_px: int, erosion_px: int,
                           progress=None) -> pd.DataFrame:
    """
    Grid-scan every stem in both label sources and count measurement points.

    One row per (source, stem, region) with at least one instance;
    n_points sums over all instances of that region.
    """
    sources = [("gt", LABELS_GT), ("predicted", LABELS_PRED)]
    rows = []
    n_total = len(ALL_STEMS) * len(sources)
    done = 0
    for src_name, label_dict in sources:
        for stem in ALL_STEMS:
            mask_data = build_masks(label_dict.get(stem, []))
            scan = grid_scan(mask_data, spacing_x_px, spacing_y_px, erosion_px)
            for cls_name in ("body", "cap", "label"):
                n_inst = len(mask_data[cls_name])
                if n_inst == 0:
                    continue
                n_pts = sum(v for (cn, ii), v in scan["counts"].items()
                            if cn == cls_name)
                rows.append({
                    "source":       src_name,
                    "stem":         stem,
                    "class_name":   cls_name,
                    "n_instances":  n_inst,
                    "n_points":     int(n_pts),
                })
            done += 1
            if progress is not None and (done % 25 == 0 or done == n_total):
                progress(done, n_total)
    return pd.DataFrame(rows)


if not ALL_STEMS:
    print("No label files found.")
else:
    # --- Batch export ---
    sx_px = max(1, int(round(BATCH_SPACING_X_MM * PIXELS_PER_MM)))
    sy_px = max(1, int(round(BATCH_SPACING_Y_MM * PIXELS_PER_MM)))
    er_px = int(round(BATCH_EROSION_MM * PIXELS_PER_MM))
    print(f"Batch run: grid {BATCH_SPACING_X_MM:.1f}×{BATCH_SPACING_Y_MM:.1f} mm "
          f"({sx_px}×{sy_px} px), erosion {BATCH_EROSION_MM:.1f} mm ({er_px} px)")
    print(f"Processing {len(ALL_STEMS)} stems × 2 sources ...")

    def _progress(done, total):
        print(f"  processed {done}/{total} image-source combinations...")

    df = batch_grid_pointcounts(sx_px, sy_px, er_px, progress=_progress)
    df["spacing_x_mm"] = BATCH_SPACING_X_MM
    df["spacing_y_mm"] = BATCH_SPACING_Y_MM
    df["erosion_mm"]   = BATCH_EROSION_MM

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = RESULTS_DIR / f"{PROJECT_NAME}_grid_pointcounts_{ts}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved point-count table ({len(df)} rows): {csv_path}")

    # --- Statistics from the saved CSV ---
    df = pd.read_csv(csv_path)
    stats = (
        df.groupby(["class_name", "source"])["n_points"]
          .agg(n_images="count", mean="mean", std="std",
               median="median", min="min", max="max")
          .round(1)
    )
    print("\nPoints per image and region:")
    print(stats.to_string())

    print("\nGT vs predicted mean point count:")
    means = df.groupby(["class_name", "source"])["n_points"].mean().unstack("source")
    means["delta_pct"] = (100 * (means["predicted"] - means["gt"]) / means["gt"]).round(1)
    print(means.round(1).to_string())

    # --- Shared subtitle and legend handles ---
    _subtitle = (f"grid {BATCH_SPACING_X_MM:.1f}×{BATCH_SPACING_Y_MM:.1f} mm, "
                 f"erosion {BATCH_EROSION_MM:.1f} mm")

    _legend_handles = [
        Patch(facecolor=C_BODY,  alpha=0.85, edgecolor="black", lw=0.3, label="Body"),
        Patch(facecolor=C_CAP,   alpha=0.85, edgecolor="black", lw=0.3, label="Cap"),
        Patch(facecolor=C_LABEL, alpha=0.85, edgecolor="black", lw=0.3, label="Label"),
        Patch(facecolor="grey",  alpha=0.85, edgecolor="black", lw=0.3, label="Predicted"),
        Patch(facecolor="grey",  alpha=0.65, hatch="///",
              edgecolor="white", lw=0.5,     label="GT"),
        Line2D([0], [0], color="black", lw=1.3, linestyle="-",  label="Median"),
        Line2D([0], [0], color="black", lw=1.1, linestyle="--", label="Mean"),
    ]

    # --- Figure 1: boxplot ---
    REGIONS = ["body", "cap", "label"]
    SRCS    = ["gt", "predicted"]

    fig, ax = plt.subplots(figsize=(_FW, _FW * 0.5))

    data, positions, box_colors, box_srcs, tickpos = [], [], [], [], []
    pos = 0
    for r in REGIONS:
        for s in SRCS:
            vals = df[(df.class_name == r) & (df.source == s)]["n_points"].values
            data.append(vals if len(vals) else np.array([np.nan]))
            positions.append(pos)
            box_colors.append(region_color[r])
            box_srcs.append(s)
            pos += 1
        tickpos.append(pos - 1.5)
        pos += 0.8

    bp = ax.boxplot(data, positions=positions, widths=0.7, patch_artist=True,
                    showfliers=False, showmeans=True, meanline=True,
                    medianprops=dict(color="black", lw=1.3, linestyle="-"),
                    meanprops=dict(color="black",   lw=1.1, linestyle="--"))

    for patch, c, s in zip(bp["boxes"], box_colors, box_srcs):
            patch.set_facecolor(c)
            if s == "gt":
                patch.set_alpha(0.65)
                patch.set_hatch("///")
                patch.set_edgecolor("white")   # white keeps hatch lines white
                patch.set_linewidth(0.5)
            else:
                patch.set_alpha(0.85)
                patch.set_edgecolor("black")   # as in the performance figure
                patch.set_linewidth(0.8)

    ax.set_xticks(tickpos)
    ax.set_xticklabels(REGIONS)
    ax.set_ylabel("Point Count")
    ax.set_title(f"Grid Point Count per Region\n{_subtitle}")
    ax.legend(handles=_legend_handles, fontsize=_FONT_SIZE - 2, ncols=4)
    fig.tight_layout()
    png_path = RESULTS_DIR / f"{PROJECT_NAME}_grid_pointcounts_{ts}.png"
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    print(f"\nSaved boxplot: {png_path}")
    plt.show()

    # --- Figure 2: split violin ---
    def _half_violin(axx, vals, center, side, color, is_gt=False):
        """Half violin in region colour; hatched for GT, solid for predicted."""
        if len(vals) == 0:
            return
        parts = axx.violinplot([vals], positions=[center], widths=0.9,
                               showextrema=False)
        for body in parts["bodies"]:
            verts = body.get_paths()[0].vertices
            if side == "left":
                verts[:, 0] = np.minimum(verts[:, 0], center)
            else:
                verts[:, 0] = np.maximum(verts[:, 0], center)
            body.set_facecolor(color)
            if is_gt:
                body.set_alpha(0.65); body.set_hatch("///")
                body.set_edgecolor("white")    # white keeps hatch lines white
                body.set_linewidth(0.5)
            else:
                body.set_alpha(0.85)
                body.set_edgecolor("#333")     # as in the performance figure
                body.set_linewidth(0.6)
        med_h  = float(np.median(vals))
        mean_h = float(np.mean(vals))
        x0, x1 = (center - 0.42, center) if side == "left" else (center, center + 0.42)
        axx.hlines(med_h,  x0, x1, color="black", lw=1.3, zorder=5)
        axx.hlines(mean_h, x0, x1, color="black", lw=1.1, ls="--", zorder=5)

    fig2, ax2 = plt.subplots(figsize=(_FW, _FW * 0.5))
    centers = np.arange(len(REGIONS))
    for cx, r in zip(centers, REGIONS):
        gt_vals   = df[(df.class_name == r) & (df.source == "gt")]["n_points"].values
        pred_vals = df[(df.class_name == r) & (df.source == "predicted")]["n_points"].values
        _half_violin(ax2, gt_vals,   cx, "left",  region_color[r], is_gt=True)
        _half_violin(ax2, pred_vals, cx, "right", region_color[r], is_gt=False)

    ax2.set_xticks(centers)
    ax2.set_xticklabels(REGIONS)
    ax2.set_ylabel("Point Count")
    ax2.set_title(f"Grid Point Count per Region\n{_subtitle}")
    ax2.legend(handles=_legend_handles, fontsize=_FONT_SIZE - 2, ncols=4)
    fig2.tight_layout()
    png_path_v = RESULTS_DIR / f"{PROJECT_NAME}_grid_pointcounts_{ts}_violin.png"
    fig2.savefig(png_path_v, dpi=150, bbox_inches="tight")
    print(f"Saved split-violin: {png_path_v}")
    plt.show()